In [1]:
function range(start: number, stop: number): number[] {
    return Array.from({ length: stop - start + 1 }, (_, index) => start + index);
}

In [2]:
import { CSP, solve, Assignment, Variable, Formula } from "./02-Backtracking-Constraint-Solver";

# Saving the Infidels

In this notebook we want so solve a famous search problem, which is usually known as the
[missionaries and cannibals problem](https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem):
Three missinaries and three infidels have to cross a river in order to get to a church where the infidels can be baptized.  In order to cross the river, they have to take a small boat that can take at most two passengers.  If at any moments at any shore there are more infidels than missionaries, then the missionaries have a problem, since the infidels have a diet that includes human flesh.

We will encode this problem as a *constraint satisfaction problem*.  In order to do so, we assume that the
problem can be solved with $n\in\mathbb{N}$ crossing of the river.  We use the following variables:
* $\texttt{M}i$ for $i\in\{0,\cdots,n\}$ is the number of missionaries on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{C}i$ for $i\in\{0,\cdots,n\}$ is the number of infidels on the western shore after the 
  $i^{\textrm{th}}$ crossing.
* $\texttt{B}i$ for $i\in\{0,\cdots,n\}$ is the number of boats on the western shore after the 
  $i^{\textrm{th}}$ crossing.

## Auxiliary Functions

The following functions generate the *formulas* required by the constraint solver. This ensures that the logical expressions can be evaluated dynamically using the variable names.

The invariant is given by the following formula:
$$ M = 0 \vee M = 3 \vee M = C $$

In [3]:
function invariant(M: Variable, C: Variable, B: Variable): Formula {
    return `${M} == 0 || ${M} == 3 || ${M} == ${C}`;
}

The formula for the transition has three components:
* The boat changes sides with every transition:
  $$ B_\beta = 1 - B_\alpha $$
* If the boat goes from the west to the east, there are two conditions:
  + The number of people on the boat must be at least 1 and at most two.
    $$ 1 \leq (M_\alpha - M_\beta) + (C_\alpha - C_\beta) \leq 2 $$
  + The number of both missionaries and infidels on the western side must not increase.
    $$ M_\beta \leq M_\alpha \wedge C_\beta \leq C_\alpha $$
* If the boat goes from the east to the west, there are two conditions:
  + The number of people on the boat must be at least 1 and at most two.
    $$ 1 \leq (M_\beta - M_\alpha) + (C_\beta - C_\alpha) \;\wedge\; 
              (M_\beta - M_\alpha) + (C_\beta - C_\alpha)\leq 2 $$
  + The number of both missionaries and infidels on the western side must not decrease.
    $$ M_\beta \geq M_\alpha \wedge C_\beta \geq C_\alpha $$    

In [4]:
function transition(M𝛼: Variable, C𝛼: Variable, B𝛼: Variable, 
                    M𝛽: Variable, C𝛽: Variable, B𝛽: Variable): Formula 
{
    const westToEast = [`(1 <= (${M𝛼} - ${M𝛽}) + (${C𝛼} - ${C𝛽})`,
                        `(${M𝛼} - ${M𝛽}) + (${C𝛼} - ${C𝛽}) <= 2`,
                        `${M𝛽} <= ${M𝛼} && ${C𝛽} <= ${C𝛼})`];
    const eastToWest = [`(1 <= (${M𝛽} - ${M𝛼}) + (${C𝛽} - ${C𝛼})`, 
                        `(${M𝛽} - ${M𝛼}) + (${C𝛽} - ${C𝛼}) <= 2`, 
                        `${M𝛽} >= ${M𝛼} && ${C𝛽} >= ${C𝛼})`];
    
    return `(${B𝛽} == 1 - ${B𝛼}) && ((${B𝛼} == 1) ? ${westToEast.join(' &&')} 
                                                  : ${eastToWest.join(' &&')})`;
}

The function `missionariesCSP` creates a CSP that tries to solve the problem with `n` crossings.

In [5]:
function missionariesCSP(n: number): CSP {
    const Vars    = range(0, n).flatMap(i => [`M${i}`, `C${i}`, `B${i}`]);
    const Values  = range(0, n);
    const Constrs = ['M0 == 3 && C0 == 3 && B0 == 1',          // Start
                     `M${n} == 0 && C${n} == 0 && B${n} == 0`, // Goal
                     ...range(0, n-1).map(i => invariant( `M${i}`, `C${i}`, `B${i}`)),
                     ...range(0, n-1).map(i => transition(`M${i}`, `C${i}`, `B${i}`, 
                                                          `M${i+1}`, `C${i+1}`, `B${i+1}`))];
    return [Vars, Values, Constrs];
}

In [6]:
missionariesCSP(3);

[
  [
    'M0', 'C0', 'B0',
    'M1', 'C1', 'B1',
    'M2', 'C2', 'B2',
    'M3', 'C3', 'B3'
  ],
  [ 0, 1, 2, 3 ],
  [
    'M0 == 3 && C0 == 3 && B0 == 1',
    'M3 == 0 && C3 == 0 && B3 == 0',
    'M0 == 0 || M0 == 3 || M0 == C0',
    'M1 == 0 || M1 == 3 || M1 == C1',
    'M2 == 0 || M2 == 3 || M2 == C2',
    '(B1 == 1 - B0) && ((B0 == 1) ? (1 <= (M0 - M1) + (C0 - C1) &&(M0 - M1) + (C0 - C1) <= 2 &&M1 <= M0 && C1 <= C0) \n' +
      '                                                  : (1 <= (M1 - M0) + (C1 - C0) &&(M1 - M0) + (C1 - C0) <= 2 &&M1 >= M0 && C1 >= C0))',
    '(B2 == 1 - B1) && ((B1 == 1) ? (1 <= (M1 - M2) + (C1 - C2) &&(M1 - M2) + (C1 - C2) <= 2 &&M2 <= M1 && C2 <= C1) \n' +
      '                                                  : (1 <= (M2 - M1) + (C2 - C1) &&(M2 - M1) + (C2 - C1) <= 2 &&M2 >= M1 && C2 >= C1))',
    '(B3 == 1 - B2) && ((B2 == 1) ? (1 <= (M2 - M3) + (C2 - C3) &&(M2 - M3) + (C2 - C3) <= 2 &&M3 <= M2 && C3 <= C2) \n' +
      '                              

The function `findSolution` computes a solution to the problem of saving the infidels by iteratively increasing the number of crossings `n`.

In [7]:
function findSolution(): [number, Assignment] {
    let n = 1;
    while (true) {
        console.log(`Checking n = ${n}...`);
        const csp = missionariesCSP(n);
        const result = solve(csp);
        if (result != null) { return [n, result]; }
        n += 2;
    }
}

Finding the solution takes about 4 seconds on my Mac M2 Max Studio.

In [8]:
console.time("findSolution"); 
const result = findSolution(); 
console.timeEnd("findSolution");
result

Checking n = 1...
Checking n = 3...
Checking n = 5...
Checking n = 7...
Checking n = 9...
Checking n = 11...
findSolution: 4.052s
[
  11,
  RecursiveMap(36) {
    B0 => 1,
    B1 => 0,
    B10 => 1,
    B11 => 0,
    B2 => 1,
    B3 => 0,
    B4 => 1,
    B5 => 0,
    B6 => 1,
    B7 => 0,
    B8 => 1,
    B9 => 0,
    C0 => 3,
    C1 => 2,
    C10 => 2,
    C11 => 0,
    C2 => 2,
    C3 => 0,
    C4 => 1,
    C5 => 1,
    C6 => 2,
    C7 => 2,
    C8 => 3,
    C9 => 1,
    M0 => 3,
    M1 => 2,
    M10 => 0,
    M11 => 0,
    M2 => 3,
    M3 => 3,
    M4 => 3,
    M5 => 1,
    M6 => 2,
    M7 => 0,
    M8 => 0,
    M9 => 0
  }
]


## Visualizing the Result

In [ ]:
function showSolution(solution: Assignment, n: number) {
    for (let i = 0; i <= n; i++) {
        const M = Number(solution.get(`M${i}`));
        const C = Number(solution.get(`C${i}`));
        const B = Number(solution.get(`B${i}`));
        
        const leftSide = '😇'.repeat(M) + '🥷'.repeat(C);
        const rightSide = '😇'.repeat(3 - M) + '🥷'.repeat(3 - C);
        const gap = " ".repeat(28 - (M + C) * 2);
        
        console.log(`${leftSide}${gap}${rightSide}`);
        
        if (i < n) {
            const nextM = Number(solution.get(`M${i+1}`));
            const nextC = Number(solution.get(`C${i+1}`));
            
            if (B == 1) {
                const MB = M - nextM;
                const CB = C - nextC;
                console.log(' '.repeat(10) + `>>> ${'😇'.repeat(MB)}${'🥷'.repeat(CB)} >>>`);
            } else {
                const MB = nextM - M;
                const CB = nextC - C;
                console.log(' '.repeat(10) + `<<< ${'😇'.repeat(MB)}${'🥷'.repeat(CB)} <<<`);
            }
        }
    }
}

In [ ]:
const [n, solution] = result;
showSolution(solution, n);